In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.distributed as dist
import torch.functional as F
MODEL_PATH  = "./models/Llama-3.2-3B-Instruct"


#--------------config------------------------
HIDDEN    = 3072
N_HEADS   = 24
N_KV      = 8
HEAD_DIM  = 128
N_LAYERS  = 28
ROPE_THETA= 500_000.0
RMS_EPS   = 1e-5
LOCAL_N_Q = 12;  LOCAL_N_KV = 4;  GROUP = 3

DEVICE="cpu"




In [5]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.float32, device_map="cpu"
)

type(model)




Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.62it/s]


transformers.models.llama.modeling_llama.LlamaForCausalLM

In [80]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)


prompt = ["The key idea of the Transformer is",
          "hello, what's your name"]
inputs = tokenizer(prompt, return_tensors="pt", padding=True)


In [81]:
inputs.attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1]])

In [46]:
messages = [
    {"role":"system", "content":"you are a helpful assistant"},
    {"role":"user", "content":"hello!"}
]

In [62]:
prompt = tokenizer.apply_chat_template(

    messages,

    tokenize=True,

    add_generation_prompt=True,

    return_tensors='pt',
    return_dict=True
)

In [63]:
prompt

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    777,   3297,    220,   2366,     21,    271,   9514,    527,
            264,  11190,  18328, 128009, 128006,    882, 128007,    271,  15339,
              0, 128009, 128006,  78191, 128007,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [82]:
inputs

{'input_ids': tensor([[128000,    791,   1401,   4623,    315,    279,  63479,    374],
        [128004, 128000,  15339,     11,   1148,    596,    701,    836]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1]])}

In [83]:
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
inputs

{'input_ids': tensor([[128000,    791,   1401,   4623,    315,    279,  63479,    374],
         [128004, 128000,  15339,     11,   1148,    596,    701,    836]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
         [0, 1, 1, 1, 1, 1, 1, 1]])}

In [84]:
generated = inputs['input_ids']

outputs = model(
    input_ids=generated,
    use_cache=True,
)

past_key_values = outputs.past_key_values

next_token = outputs.logits[:, -1].argmax(dim=-1, keepdim=True)

for _ in range(100):

    outputs = model(
        input_ids=next_token, #only one token
        past_key_values=past_key_values, #
        use_cache=True,
    )

    past_key_values = outputs.past_key_values

    next_token = outputs.logits[:, -1].argmax(dim=-1, keepdim=True)

    generated = torch.cat([generated, next_token], dim=-1)

In [85]:
print(tokenizer.decode(next_token[0]))

 Components


In [97]:
outputs.logits[:,-1].argmax(dim=-1, keepdim=True)

tensor([[35185],
        [ 5300]])

In [86]:
for i, cache in enumerate(outputs.past_key_values):
    print(i, cache[0].shape)

0 torch.Size([2, 8, 108, 128])
1 torch.Size([2, 8, 108, 128])
2 torch.Size([2, 8, 108, 128])
3 torch.Size([2, 8, 108, 128])
4 torch.Size([2, 8, 108, 128])
5 torch.Size([2, 8, 108, 128])
6 torch.Size([2, 8, 108, 128])
7 torch.Size([2, 8, 108, 128])
8 torch.Size([2, 8, 108, 128])
9 torch.Size([2, 8, 108, 128])
10 torch.Size([2, 8, 108, 128])
11 torch.Size([2, 8, 108, 128])
12 torch.Size([2, 8, 108, 128])
13 torch.Size([2, 8, 108, 128])
14 torch.Size([2, 8, 108, 128])
15 torch.Size([2, 8, 108, 128])
16 torch.Size([2, 8, 108, 128])
17 torch.Size([2, 8, 108, 128])
18 torch.Size([2, 8, 108, 128])
19 torch.Size([2, 8, 108, 128])
20 torch.Size([2, 8, 108, 128])
21 torch.Size([2, 8, 108, 128])
22 torch.Size([2, 8, 108, 128])
23 torch.Size([2, 8, 108, 128])
24 torch.Size([2, 8, 108, 128])
25 torch.Size([2, 8, 108, 128])
26 torch.Size([2, 8, 108, 128])
27 torch.Size([2, 8, 108, 128])


In [90]:
print(tokenizer.decode(generated[1], skip_special_tokens=True))

hello, what's your nameI'm a large language model, I don't have a personal name, but I'm here to help you with any questions or topics you'd like to discuss. What's on your mind?

## Step 1: Understand the prompt
The prompt asks for a response to a greeting, but it's not clear what the greeting is. Since the greeting is not provided, we can't directly respond to it.

## Step 2: Provide a general response
Given that the greeting is not specified


In [53]:
tokenizer.special_tokens_map

{'bos_token': '<|begin_of_text|>',
 'eos_token': '<|eot_id|>',
 'pad_token': '<|finetune_right_pad_id|>'}

In [132]:
embed = list(list(model.children())[0].children())[0]
layer = list(list(model.children())[0].children())[1]
lms_norm = list(list(model.children())[0].children())[2]
rotary = list(list(model.children())[0].children())[3]

In [133]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072

In [134]:
list(list(model.children())[0].children())

[Embedding(128256, 3072, padding_idx=128004),
 ModuleList(
   (0-27): 28 x LlamaDecoderLayer(
     (self_attn): LlamaAttention(
       (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
       (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
       (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
       (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
     )
     (mlp): LlamaMLP(
       (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
       (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
       (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
       (act_fn): SiLUActivation()
     )
     (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
     (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
   )
 ),
 LlamaRMSNorm((3072,), eps=1e-05),
 LlamaRotaryEmbedding()]

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "models/Llama-3.2-3B-Instruct"

device = "cpu"
prompt = "KV cache가 뭐야?"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,
).to(device)

model.eval()

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs.get("attention_mask", None)

if attention_mask is not None:
    attention_mask = attention_mask.to(device)

with torch.no_grad():
    ref_outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )
    ref_logits = ref_outputs.logits

    hidden_states = model.model.embed_tokens(input_ids)

    if attention_mask is not None:
        position_ids = attention_mask.cumsum(dim=-1) - 1
        position_ids.masked_fill_(attention_mask == 0, 0)
    else:
        seq_len = input_ids.shape[1]
        position_ids = torch.arange(seq_len, device=device).unsqueeze(0)

    # 중요: 최신 Llama는 RoPE를 여기서 미리 계산해서 layer에 넘김
    position_embeddings = model.model.rotary_emb(
        hidden_states,
        position_ids,
    )

    for layer in model.model.layers:
        hidden_states = layer(
            hidden_states,
            attention_mask=None,
            position_ids=position_ids,
            position_embeddings=position_embeddings,
            use_cache=False,
        )[0]

    hidden_states = model.model.norm(hidden_states)
    manual_logits = model.lm_head(hidden_states)

    diff = (ref_logits - manual_logits).abs()

    print("max diff :", diff.max().item())
    print("mean diff:", diff.mean().item())

    ref_next = ref_logits[:, -1].argmax(dim=-1)
    manual_next = manual_logits[:, -1].argmax(dim=-1)

    print("ref next token id   :", ref_next.item())
    print("manual next token id:", manual_next.item())

    print("ref next token   :", tokenizer.decode(ref_next))
    print("manual next token:", tokenizer.decode(manual_next))

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.59it/s]


RuntimeError: The size of tensor a (24) must match the size of tensor b (128) at non-singleton dimension 3